# Composite scoring validation

Starting point: reuse the real `postprocess/composite_scoring` code (metric definitions, thresholds, domains, scoring logic) against the actual cohort data in this folder, instead of re-deriving anything from scratch.

Step 1 for now: get the 8 metrics out of the raw files and check how correlated they actually are with each other.

In [ ]:
import sys
from pathlib import Path

# src-layout package root for the AngioEye repo (postprocess/, pipelines/, input_output/ live under here)
REPO_SRC = Path("/Users/admin/Developer/AngioEye/src")
if str(REPO_SRC) not in sys.path:
    sys.path.insert(0, str(REPO_SRC))

import h5py
import numpy as np
import pandas as pd

from pipelines import load_pipeline_catalog
from pipeline_engine.execution import run_pipeline_file
from input_output.output_paths import h5_output_dir
from input_output.hdf5_schema import find_pipeline_group
from input_output.hdf5_io import read_dataset

from postprocess.composite_scoring.metrics import (
    METRICS,
    DOMAINS,
    PLOT_VESSEL_TYPE,
    VESSEL_TYPES,
    REPRESENTATIONS,
)
from postprocess.composite_scoring.scoring import (
    append_scores_to_file,
    score_records_for_tree,
    _finite_scalar,
)

DATA_ROOT = Path("/Users/admin/Desktop/langevin-internship/composite-scoring/WAS scoring")
PROCESSED_ROOT = DATA_ROOT / "_processed"

# This exploration is restricted to the bandlimited representation throughout
# (see metrics.py's REPRESENTATIONS for the full set) -- raw is not pulled in.
REPRESENTATION = "bandlimited"
assert REPRESENTATION in REPRESENTATIONS

list(METRICS.keys())


## 1. Discover cohort files

Each cohort folder here has two subfolders (the two groups being compared: control vs pathological, before vs after, etc). This mirrors exactly how `extract_group_name` assigns `cohort` in the real pipeline (`postprocess/core/grouped_batch.py`) — first path component under the batch root.

In [2]:
def discover_cohort_files(root: Path) -> dict[str, dict[str, list[Path]]]:
    manifest: dict[str, dict[str, list[Path]]] = {}
    for cohort_dir in sorted(root.iterdir()):
        if not cohort_dir.is_dir():
            continue
        if cohort_dir.name in {"zips", "_processed"} or cohort_dir.name.startswith("."):
            continue
        groups: dict[str, list[Path]] = {}
        for group_dir in sorted(cohort_dir.iterdir()):
            if not group_dir.is_dir():
                continue
            files = sorted(group_dir.glob("*.h5"))
            if files:
                groups[group_dir.name] = files
        if groups:
            manifest[cohort_dir.name] = groups
    return manifest


manifest = discover_cohort_files(DATA_ROOT)
for cohort, groups in manifest.items():
    print(cohort, {group: len(files) for group, files in groups.items()})


251219_BOM0753_no_BL2 {'BL': 7, 'F': 12}
260609_BOM0753 Valsalva {'BL': 6, 'Valsalva_2': 4}
260609_GOA Valsalva {'BL': 4, 'Valsalva': 5}
260609_GUJ Valsalva {'BL': 4, 'Valsalva': 9}
GLONGI_select {'after': 17, 'before': 17}
POAG_vs_CTRL {'CTRL': 262, 'POAG': 245}
alternate_pressure {'ctrl': 13, 'pressure': 13}


## 2. Compute waveform_shape_metrics (separate block)

None of the raw files have the `waveform_shape_metrics` pipeline group yet, so the 8 metrics don't exist in them directly — this runs the real `waveform_shape_metrics` pipeline (the same one `postprocess/composite_scoring` reads from) via `pipeline_engine.execution.run_pipeline_file`, the same function the CLI/app use.

Idempotent: skips any file whose processed copy already exists in `_processed/`, so re-running this cell after the first pass is fast. ~600 files total across all cohorts, a few minutes on first run.

In [3]:
available_pipelines, _ = load_pipeline_catalog()
pipeline_registry = {p.name: p for p in available_pipelines}
wsm_pipeline = pipeline_registry["waveform_shape_metrics"]


def processed_path_for(raw_path: Path) -> Path:
    relative_parent = raw_path.parent.relative_to(DATA_ROOT)
    return h5_output_dir(PROCESSED_ROOT) / relative_parent / f"{raw_path.stem}_wsm.h5"


def ensure_waveform_shape_metrics(raw_path: Path) -> Path | None:
    expected = processed_path_for(raw_path)
    if expected.exists():
        return expected
    relative_parent = raw_path.parent.relative_to(DATA_ROOT)
    return run_pipeline_file(
        raw_path,
        [wsm_pipeline],
        PROCESSED_ROOT,
        output_relative_parent=relative_parent,
        output_filename=f"{raw_path.stem}_wsm.h5",
    )


# Real acquisitions are messy -- some files won't have everything the pipeline
# expects (e.g. a missing Artery/VelocityPerBeat group). Skip and record those
# rather than aborting the whole batch, mirroring how run_composite_scoring
# collects per-file failures instead of stopping on the first one.
pipeline_failures: list[str] = []
processed_manifest: dict[str, dict[str, list[Path]]] = {}
for cohort, groups in manifest.items():
    processed_manifest[cohort] = {}
    for group, files in groups.items():
        processed_paths: list[Path] = []
        for raw_path in files:
            try:
                processed_paths.append(ensure_waveform_shape_metrics(raw_path))
            except Exception as exc:  # noqa: BLE001
                pipeline_failures.append(
                    f"{cohort}/{group}/{raw_path.name}: {type(exc).__name__}: {exc}"
                )
        processed_manifest[cohort][group] = processed_paths
        print(f"{cohort}/{group}: {len(processed_paths)}/{len(files)} file(s) ready")

if pipeline_failures:
    print(f"\n{len(pipeline_failures)} file(s) skipped:")
    for failure in pipeline_failures:
        print(f"  {failure}")


251219_BOM0753_no_BL2/BL: 7/7 file(s) ready
251219_BOM0753_no_BL2/F: 10/12 file(s) ready
260609_BOM0753 Valsalva/BL: 6/6 file(s) ready
260609_BOM0753 Valsalva/Valsalva_2: 4/4 file(s) ready
260609_GOA Valsalva/BL: 4/4 file(s) ready
260609_GOA Valsalva/Valsalva: 4/5 file(s) ready
260609_GUJ Valsalva/BL: 2/4 file(s) ready
260609_GUJ Valsalva/Valsalva: 6/9 file(s) ready
GLONGI_select/after: 12/17 file(s) ready
GLONGI_select/before: 15/17 file(s) ready
POAG_vs_CTRL/CTRL: 260/262 file(s) ready
POAG_vs_CTRL/POAG: 241/245 file(s) ready
alternate_pressure/ctrl: 13/13 file(s) ready
alternate_pressure/pressure: 13/13 file(s) ready

21 file(s) skipped:
  251219_BOM0753_no_BL2/F/251219_BOM0753_15_HD_1_EF_1_output.h5: RuntimeError: Pipeline 'waveform_shape_metrics' failed: KeyError: 'Unable to synchronously open object (component not found)'
  at /Users/admin/Developer/AngioEye/src/pipelines/waveform_shape_metrics.py:1581 in run()
    T = np.asarray(h5file[self.T_input])
    ^
  251219_BOM0753_no_BL

## 3. Append composite scores + pull out the raw metric values

`append_scores_to_file` is the exact production entrypoint (writes RWAS/RWAS4 into the `composite_scoring` group) — running it here so the processed files match what the real pipeline would produce.

For the correlation check we need the 8 raw metric values themselves, not just the collapsed RWAS/RWAS4 score. `_build_scores_tree` in `scoring.py` doesn't expose those (it only keeps the final domain-collapsed score), so `extract_metric_values` below re-does just that one read step — same `Metric.path()` / `Metric.derived_paths()` / `_finite_scalar` used internally, just returning the per-metric scalar instead of folding it into RWAS. Only the `bandlimited` representation is read (see `REPRESENTATION` above).

In [ ]:
def extract_metric_values(
    processed_path: Path, vessel_type: str = PLOT_VESSEL_TYPE
) -> dict[str, float | None]:
    values: dict[str, float | None] = {key: None for key in METRICS}
    with h5py.File(processed_path, "r") as h5:
        source_group = find_pipeline_group(h5, "waveform_shape_metrics")
        if source_group is None:
            return values

        for metric_key, metric in METRICS.items():
            paths = metric.derived_paths(vessel_type, REPRESENTATION)
            if paths is None:
                raw = read_dataset(
                    source_group, metric.path(vessel_type, REPRESENTATION), default=None
                )
                value = _finite_scalar(raw) if raw is not None else None
            else:
                numerator = read_dataset(source_group, paths[0], default=None)
                denominator = read_dataset(source_group, paths[1], default=None)
                if numerator is None or denominator is None:
                    value = None
                else:
                    numerator = np.asarray(numerator, dtype=float)
                    denominator = np.asarray(denominator, dtype=float)
                    with np.errstate(divide="ignore", invalid="ignore"):
                        ratio = np.where(
                            np.isfinite(denominator) & (denominator != 0),
                            numerator / denominator,
                            np.nan,
                        )
                    value = _finite_scalar(ratio)
            values[metric_key] = value
    return values


rows = []
for cohort, groups in processed_manifest.items():
    for group, paths in groups.items():
        for path in paths:
            # Keep the real pipeline exercised end to end (RWAS/RWAS4 written to file).
            append_scores_to_file(path)
            metric_values = extract_metric_values(path)
            for metric_key in METRICS:
                rows.append(
                    {
                        "cohort": cohort,
                        "group": group,
                        "file": path.name,
                        "metric": metric_key,
                        "value": metric_values[metric_key],
                    }
                )

long_df = pd.DataFrame(rows)
wide_df = long_df.pivot_table(
    index=["cohort", "group", "file"],
    columns="metric",
    values="value",
).reset_index()
wide_df.head()


## 4. Correlation between the 8 metrics

Pearson (linear) and Spearman (rank, less sensitive to the ratio-metric distributions) correlation across all cohorts pooled, computed on the `bandlimited` representation only (see `REPRESENTATION` in section imports).

`metric -> domain` is printed alongside so it's easy to check whether metrics the current code groups into the same domain (`timing`, `spectral`, `persistence`, `pulsatility` in `metrics.py`) actually correlate with each other more than with metrics from other domains.

In [ ]:
metric_cols = list(METRICS.keys())

metric_to_domain = {
    metric_key: domain_name
    for domain_name, domain in DOMAINS.items()
    for metric_key in domain.metrics
}
print("metric -> domain:")
for metric_key in metric_cols:
    print(f"  {metric_key:32s} {metric_to_domain[metric_key]}")

subset = wide_df[metric_cols].dropna()
pearson_corr = subset.corr(method="pearson")
spearman_corr = subset.corr(method="spearman")
n_complete = len(subset)

print(f"\n--- bandlimited (n={n_complete} complete rows) ---")
print("Pearson:")
print(pearson_corr.round(2))
print("Spearman:")
print(spearman_corr.round(2))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pearson_corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(metric_cols)))
ax.set_xticklabels(metric_cols, rotation=90)
ax.set_yticks(range(len(metric_cols)))
ax.set_yticklabels(metric_cols)
ax.set_title(f"Pearson correlation (bandlimited, n={n_complete})")
for i in range(len(metric_cols)):
    for j in range(len(metric_cols)):
        ax.text(j, i, f"{pearson_corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()


## 5. The 7-metric cluster in isolation

Section 4's full 8x8 heatmap has `low_freq_spectral_fraction` sitting in it too, which dilutes the color scale and makes the 7-metric redundancy harder to read at a glance. Drop the outlier and replot just `stroke_fraction`, `med_displacement_timing`, `late_cycle_mean_fraction`, `participation_ratio_eff_supp`, `resistivity_index`, `pulsatility_index`, `near_peak_crest_width` against each other (bandlimited only) to make the redundancy the headline instead of something you have to search for in the corner of an 8x8 grid.

In [ ]:
cluster_cols = [m for m in metric_cols if m != "low_freq_spectral_fraction"]

fig, ax = plt.subplots(figsize=(6, 5))
corr = pearson_corr.loc[cluster_cols, cluster_cols]
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(cluster_cols)))
ax.set_xticklabels(cluster_cols, rotation=90)
ax.set_yticks(range(len(cluster_cols)))
ax.set_yticklabels(cluster_cols)
ax.set_title(
    f"7-metric cluster only (bandlimited, n={n_complete})\n"
    "low_freq_spectral_fraction excluded"
)
for i in range(len(cluster_cols)):
    for j in range(len(cluster_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

off_diag = corr.values[~np.eye(len(cluster_cols), dtype=bool)]
print(f"bandlimited: min |r| = {abs(off_diag).min():.2f}, median |r| = {np.median(abs(off_diag)):.2f}, max |r| = {abs(off_diag).max():.2f}")

## 6. Correlation split by cohort/group

Section 4's pooled correlation mixes true metric redundancy with disease-driven co-movement: two metrics can look correlated across a mixed CTRL+pathology sample just because both shift with disease status, even if they aren't related within a single homogeneous population. Recompute the 7-metric cluster's correlation separately within each `(cohort, group)` pair (e.g. `POAG_vs_CTRL/CTRL` alone, `POAG_vs_CTRL/POAG` alone) instead of pooling everything, on the `bandlimited` representation.

Most of these groups are small (single digits to twenties) — only `POAG_vs_CTRL/CTRL` (n≈260) and `POAG_vs_CTRL/POAG` (n≈240) are large enough for a low-noise within-group correlation estimate on their own. The rest are reported for context but should be read as noisy.

In [ ]:
cluster_cols = [m for m in metric_cols if m != "low_freq_spectral_fraction"]

cohort_rows = []
for (cohort, group), sub in wide_df.groupby(["cohort", "group"]):
    subset = sub[cluster_cols + ["low_freq_spectral_fraction"]].dropna()
    n = len(subset)
    if n < 3:
        continue
    cluster_corr = subset[cluster_cols].corr(method="pearson").values
    off_diag = cluster_corr[~np.eye(len(cluster_cols), dtype=bool)]
    spectral_corr = subset[cluster_cols].corrwith(subset["low_freq_spectral_fraction"])
    cohort_rows.append({
        "cohort": cohort,
        "group": group,
        "n": n,
        "cluster_median_abs_r": np.median(np.abs(off_diag)),
        "cluster_min_abs_r": np.abs(off_diag).min(),
        "cluster_max_abs_r": np.abs(off_diag).max(),
        "spectral_mean_abs_r": spectral_corr.abs().mean(),
    })

cohort_corr_df = pd.DataFrame(cohort_rows).sort_values(["n"], ascending=False).reset_index(drop=True)
pd.set_option("display.max_rows", None)
cohort_corr_df.round(2)

Plotting the 7-metric cluster's median |r| per `(cohort, group)`, against the pooled reference line from section 5, to see whether the redundancy holds up within homogeneous groups or was inflated by pooling.

In [ ]:
pooled_median = np.median(np.abs(
    pearson_corr.loc[cluster_cols, cluster_cols].values[~np.eye(len(cluster_cols), dtype=bool)]
))

fig, ax = plt.subplots(figsize=(9, 5))
labels = cohort_corr_df["cohort"] + " / " + cohort_corr_df["group"] + " (n=" + cohort_corr_df["n"].astype(str) + ")"
ax.barh(labels, cohort_corr_df["cluster_median_abs_r"], color="#4C72B0")
ax.axvline(pooled_median, color="crimson", linestyle="--", label=f"pooled median = {pooled_median:.2f}")
ax.set_xlim(0, 1)
ax.set_xlabel("median |r| within 7-metric cluster")
ax.set_title("bandlimited")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()